# 01 · Fetch municipal KPIs from Kolada

**Source:** [Kolada](https://www.kolada.se) — Swedish municipal key performance indicators (RKA). Free, no API key.
**API docs:** https://github.com/Hypergene/kolada

| KPI | Description |
|---|---|
| `N01951` | Population, 31 December |
| `N02280` | Registered unemployed (Arbetsförmedlingen, November), % of labour force aged 20–64. Only available 2020–2024 |

Output (unchanged API responses, flattened) → `data_raw/`

In [ ]:
import time

import pandas as pd
import requests

## Settings
Years 2016–2025 are requested. The analysis period (2020–2024) is decided later in `03_clean_merge.ipynb`, so the raw files keep everything the API returned.

In [ ]:
BASE_URL = "https://api.kolada.se/v3"
YEARS = ",".join(str(y) for y in range(2016, 2026))   # "2016,2017,...,2025"

## A reusable fetch function

Kolada returns at most 5,000 rows per response. If there is more data, the response contains a `next_url`.
The loop keeps following `next_url` until it is empty (pagination).

Each row is nested: one object per municipality-year, with a `values` list holding one value per gender
(`K` = women, `M` = men, `T` = total). `pd.json_normalize` flattens this into one row per municipality-year-gender.

In [ ]:
def fetch_kolada(kpi: str, years: str) -> pd.DataFrame:
    """Download one Kolada KPI for all municipalities and the given years."""
    all_rows = []
    url = f"{BASE_URL}/data/kpi/{kpi}/year/{years}"

    while url:
        response = requests.get(url, timeout=60)
        response.raise_for_status()          # stop loudly on any HTTP error
        data = response.json()

        all_rows.extend(data["values"])
        print(f"{kpi}: {len(all_rows):,} rows so far")

        url = data.get("next_url")           # None on the last page -> loop ends
        time.sleep(0.5)                      # be polite to the server

    return pd.json_normalize(
        all_rows,
        record_path="values",                # one row per gender value
        meta=["kpi", "period", "municipality"],
    )

## Population

In [ ]:
population = fetch_kolada("N01951", YEARS)
print(population.shape)
population.to_csv("../data_raw/kolada_population_raw.csv", index=False)

## Unemployment

In [ ]:
unemployment = fetch_kolada("N02280", YEARS)
print(unemployment.shape)
print("years available:", sorted(unemployment["period"].unique()))   # 2020–2024 only
unemployment.to_csv("../data_raw/kolada_unemployment_raw.csv", index=False)

## Municipality and region names

Used later to build the municipality dimension table.
`type`: `K` = municipality (kommun), `L` = region (län).

In [ ]:
all_rows = []
url = f"{BASE_URL}/municipality"

while url:
    response = requests.get(url, timeout=60)
    response.raise_for_status()
    data = response.json()
    all_rows.extend(data["values"])
    url = data.get("next_url")
    time.sleep(0.5)

municipalities = pd.DataFrame(all_rows)
print(municipalities["type"].value_counts())    # expect 290 K, 22 L
municipalities.to_csv("../data_raw/kolada_municipalities_raw.csv", index=False)